In [9]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.models import Sequential

In [10]:
# Opening a zip file
# from zipfile import ZipFile
# with ZipFile('Data/archive.zip', 'r') as zip_ref:
#     zip_ref.extractall('archive')

In [11]:
# Data paths
yes_data = 'Data/archive/brain_tumor_dataset/yes'
no_data = 'Data/archive/brain_tumor_dataset/no'

In [12]:
# Inspect number of images in each category
import os
num_yes = len(os.listdir(yes_data))
num_no = len(os.listdir(no_data))
print(f"Number of 'yes' images: {num_yes}")
print(f"Number of 'no' images: {num_no}")

Number of 'yes' images: 155
Number of 'no' images: 98


In [13]:
# Standardize the images
# for image in no_data:
#     img = tf.keras.preprocessing.image.load_img(image, target_size=(64, 64))
#     img.save(image)

In [ ]:
import os, random, shutil

# # === EDIT THESE THREE ===
# YES_DIR = r"Data/archive/brain_tumor_dataset/yes"      # your YES images
# NO_DIR  = r"Data/archive/brain_tumor_dataset/no"     # your NO images
# OUT_DIR = r"Data/archive/brain_tumor_just_split"   # new folder that will hold equal samples
# # =========================

# random.seed(42)
# os.makedirs(os.path.join(OUT_DIR, "yes"), exist_ok=True)
# os.makedirs(os.path.join(OUT_DIR, "no"), exist_ok=True)

# # Get all images
# yes_images = [f for f in os.listdir(YES_DIR) if f.lower().endswith(('.png','.jpg','.jpeg','.bmp','.tif','.tiff','.gif'))]
# no_images  = [f for f in os.listdir(NO_DIR)  if f.lower().endswith(('.png','.jpg','.jpeg','.bmp','.tif','.tiff','.gif'))]

# # Find equal count
# n = min(len(yes_images), len(no_images))
# print(f"Using {n} images from each class.")

# # Randomly sample equal subsets
# yes_subset = random.sample(yes_images, n)
# no_subset  = random.sample(no_images,  n)

# # Copy to new balanced folder
# for f in yes_subset:
#     shutil.copy2(os.path.join(YES_DIR, f), os.path.join(OUT_DIR, "yes", f))
# for f in no_subset:
#     shutil.copy2(os.path.join(NO_DIR, f), os.path.join(OUT_DIR, "no", f))

# print("Balanced dataset created here:", OUT_DIR)


Using 98 images from each class.
Balanced dataset created here: Data/archive/brain_tumor_just_split


In [15]:
yes_data = 'Data/archive/brain_tumor_just_split/yes'
no_data = 'Data/archive/brain_tumor_just_split/no'


num_yes = len(os.listdir(yes_data))
num_no = len(os.listdir(no_data))
print(f"Number of 'yes' images: {num_yes}")
print(f"Number of 'no' images: {num_no}")

Number of 'yes' images: 98
Number of 'no' images: 98


In [ ]:
# import math

# SRC_DIR = r"Data/archive/brain_tumor_just_split"   # contains balanced_data/yes and balanced_data/no
# OUT_DIR      = r"Data/archive/brain_tumor_prepped"
# SPLIT        = (0.70, 0.15, 0.15)  # train, val, test ratios



# random.seed(42)
# CLASSES = ["yes", "no"]
# EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tif", ".tiff")

# # make destination folders
# for split in ["train", "val", "test"]:
#     for cls in CLASSES:
#         os.makedirs(os.path.join(OUT_DIR, split, cls), exist_ok=True)

# # split and copy
# for cls in CLASSES:
#     src_folder = os.path.join(SRC_DIR, cls)
#     files = [f for f in os.listdir(src_folder)
#              if f.lower().endswith(EXTS) and os.path.isfile(os.path.join(src_folder, f))]
#     random.shuffle(files)

#     n = len(files)
#     n_train = math.floor(SPLIT[0] * n)
#     n_val   = math.floor(SPLIT[1] * n)

#     train_files = files[:n_train]
#     val_files   = files[n_train:n_train + n_val]
#     test_files  = files[n_train + n_val:]

#     for f in train_files:
#         shutil.copy2(os.path.join(src_folder, f), os.path.join(OUT_DIR, "train", cls, f))
#     for f in val_files:
#         shutil.copy2(os.path.join(src_folder, f), os.path.join(OUT_DIR, "val", cls, f))
#     for f in test_files:
#         shutil.copy2(os.path.join(src_folder, f), os.path.join(OUT_DIR, "test", cls, f))

#     print(f"{cls}: total={n} → train={len(train_files)}, val={len(val_files)}, test={len(test_files)}")

# print("Done! Your folders are ready at:", OUT_DIR)

yes: total=98 → train=68, val=14, test=16
no: total=98 → train=68, val=14, test=16
Done! Your folders are ready at: Data/archive/brain_tumor_prepped


In [22]:
# Preparing the data
train_datagen = ImageDataGenerator(rescale=1./255,
                                   shear_range=0.2,
                                   zoom_range=0.2,
                                   horizontal_flip=True)
train_set = train_datagen.flow_from_directory('Data/archive/brain_tumor_prepped/train',
                                                    target_size=(64, 64),
                                                    batch_size=32,
                                                    class_mode='binary')

Found 136 images belonging to 2 classes.


In [23]:
# 
test_datagen = ImageDataGenerator(rescale=1./255)
test_set = test_datagen.flow_from_directory('Data/archive/brain_tumor_prepped/test',
                                                  target_size=(64, 64),
                                                  batch_size=32,
                                                  class_mode='binary')

Found 32 images belonging to 2 classes.


In [24]:
# Initialize the CNN
cnn = tf.keras.models.Sequential()
# Step 1 - Convolution
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu', input_shape=[64, 64, 3]))
# Step 2 - Pooling
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))
# Adding a second convolutional layer
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu'))
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))
# Step 3 - Flattening
cnn.add(tf.keras.layers.Flatten())
# Step 4 - Full Connection
cnn.add(tf.keras.layers.Dense(units=128, activation='relu'))
# Step 5 - Output Layer
cnn.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

c:\Users\User\anaconda3\envs\learn-env\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [26]:
# Compiling the CNN
cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [27]:
# Training the CNN
cnn.fit(x=train_set, validation_data=test_set, epochs=25)


c:\Users\User\anaconda3\envs\learn-env\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/25
5/5 ━━━━━━━━━━━━━━━━━━━━ 12s 1s/step - accuracy: 0.4779 - loss: 0.7890 - val_accuracy: 0.7188 - val_loss: 0.5048
Epoch 2/25
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 314ms/step - accuracy: 0.6912 - loss: 0.6024 - val_accuracy: 0.7812 - val_loss: 0.5731
Epoch 3/25
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 335ms/step - accuracy: 0.7132 - loss: 0.5552 - val_accuracy: 0.7500 - val_loss: 0.5066
Epoch 4/25
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 462ms/step - accuracy: 0.7279 - loss: 0.5355 - val_accuracy: 0.7812 - val_loss: 0.5249
Epoch 5/25
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 464ms/step - accuracy: 0.7353 - loss: 0.5552 - val_accuracy: 0.7500 - val_loss: 0.4556
Epoch 6/25
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 620ms/step - accuracy: 0.7279 - loss: 0.5381 - val_accuracy: 0.7188 - val_loss: 0.4523
Epoch 7/25
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 393ms/step - accuracy: 0.7426 - loss: 0.5271 - val_accuracy: 0.7812 - val_loss: 0.4575
Epoch 8/25
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 313ms/step - accuracy: 0.7574 - loss: 0.5215 - val_accuracy: 0.7500 - val_loss: 0